# Qwen Concept Probe: 100 concepts × controlled semantic-distance comparators

[View GitHub experiment folder](https://github.com/ryouy/GraduationThesis/tree/main/1111_qwen_probe_100concepts)

**Goal:** Identify roughly 20 distinct target concepts whose X-vs-Y linear Probes can discriminate independently authored *new* held-out descriptions, while controlling comparator distance in the embedding space.

**Exploratory only:** This notebook uses 100 predefined concepts in 10 groups, 3 **train** descriptions and 2 **unseen evaluation** descriptions per concept. Each has two phrasing wrappers. Small evaluation-family counts make AUC noisy and cannot establish strong robustness. A later confirmatory run must use fresh, independently reviewed evaluation descriptions for winners.

**Distance:** Sentence-Transformers `all-mpnet-base-v2` on **concept names + concise concept glosses (training description only)**; cosine distance; near/mid/far comparisons chosen *within the same category*, without peeking at Probe AUC. Distance is a relative within-category rank; not a universal or calibrated semantic distance.

**Probe:** Qwen2.5-0.5B-Instruct, frozen revision, 24th-layer `assistant_start` readout, neutral instructions; L2 logistic regression per unordered X/Y pair; three leave-one-training-family-out folds; held-out-family macro AUC; label shuffle for shortlisted pairs. Probe AUC is **relative identity decodability, not absolute activation or suppression**.

**Persistence:** GitHub `GITHUB_RESULTS_TOKEN` Colab Secret (fine-grained PAT scoped to `ryouy/GraduationThesis`, Contents read/write); output in `results/20261111_qwen100_v1/`. A changing run's checkpoint requires a **new RUN_ID**; never overwrite inconsistent data.

In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" "sentence-transformers>=3,<6" pandas numpy matplotlib requests tqdm tabulate

In [ ]:
import os, io, json, re, math, time, hashlib, zipfile, random, platform, itertools
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import numpy as np, pandas as pd, torch, requests
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from google.colab import userdata

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION='7ae557604adf67be50417f59c2c2f167def9a775'
EMBEDDER_ID='sentence-transformers/all-mpnet-base-v2'
RUN_ID='20261111_qwen100_v1'
REPO='ryouy/GraduationThesis'
ROOT='1111_qwen_probe_100concepts'
OUTPUT_PREFIX=f'{ROOT}/results/{RUN_ID}'
WORK=Path('/content/qwen_100_probe')/RUN_ID
CACHE=WORK/'features'; CACHE.mkdir(parents=True,exist_ok=True)
OUT=WORK/'outputs';OUT.mkdir(parents=True,exist_ok=True)
LAYERS=[22,23,24]
POSITION='assistant_start'
BATCH_SIZE=48
UPLOAD_EVERY=5
assert torch.cuda.is_available(),'Select T4 GPU runtime in Colab.'
GITHUB_TOKEN=userdata.get('GITHUB_RESULTS_TOKEN')
assert GITHUB_TOKEN,'Create GITHUB_RESULTS_TOKEN in Colab Secrets and enable notebook access.'
HEADERS={'Authorization':f'Bearer {GITHUB_TOKEN}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
BASE=f'https://api.github.com/repos/{REPO}/contents/'
sess=requests.Session();sess.headers.update(HEADERS)

def gh_get(path):
    r=sess.get(BASE+path,timeout=90)
    if r.status_code==404:return None
    r.raise_for_status()
    obj=r.json();return obj if isinstance(obj,dict) else None

def gh_download(path):
    meta=gh_get(path)
    if meta is None:return None
    if meta.get('type')!='file':raise RuntimeError('Not a file: '+path)
    raw=sess.get(meta['download_url'],headers={'Authorization':f'Bearer {GITHUB_TOKEN}'},timeout=180)
    raw.raise_for_status()
    return raw.content

def gh_put_immutable(path, blob, message):
    remote=gh_get(path)
    newsha=hashlib.sha1(b'blob '+str(len(blob)).encode()+b'\0'+blob).hexdigest()
    if remote is not None:
        if remote['sha']!=newsha:
            raise RuntimeError('Existing path has different content; use a NEW RUN_ID: '+path)
        print('Already saved:',path);return
    import base64
    payload={'message':message,'content':base64.b64encode(blob).decode(),'branch':'main'}
    r=sess.put(BASE+path,json=payload,timeout=180)
    r.raise_for_status()
    print('GitHub saved:',path)

In [ ]:
BANK = {'elephant': {'group': 'land_mammals', 'facts': ['A massive herbivore carries a flexible trunk.', 'Large ears release heat in warm regions.', 'Families travel together under older females.', 'Thick pillars support an exceptionally heavy body.', 'Ivory tusks can grow from upper teeth.']}, 'giraffe': {'group': 'land_mammals', 'facts': ['An extremely tall browser reaches high foliage.', 'A spotted coat breaks up its silhouette.', 'Long legs make drinking from pools awkward.', 'Individuals feed from branches far above ground.', 'Newborns stand soon after a considerable fall.']}, 'lion': {'group': 'land_mammals', 'facts': ['A large social predator lives in family prides.', 'Adult males often grow a heavy mane.', 'Groups cooperate when pursuing large prey.', 'Territorial roaring travels over long distances.', 'Resting occupies much of the hot day.']}, 'tiger': {'group': 'land_mammals', 'facts': ['A solitary striped predator hunts by ambush.', 'Orange fur carries bold dark bands.', 'Swimming is common among these powerful cats.', 'Individuals occupy territories in dense habitats.', 'A silent approach precedes a sudden attack.']}, 'wolf': {'group': 'land_mammals', 'facts': ['A pack hunter communicates with howls.', 'Long-distance travel helps track hoofed prey.', 'Social groups often center on breeding adults.', 'Scent marks help establish territorial boundaries.', 'Thick seasonal fur protects against winter cold.']}, 'fox': {'group': 'land_mammals', 'facts': ['A small wild canine often hunts rodents alone.', 'A bushy tail assists with balance and warmth.', 'Some species adapt well to cities.', 'Sharp hearing locates prey below snow or grass.', 'The animal frequently rests in an underground den.']}, 'rabbit': {'group': 'land_mammals', 'facts': ['A long-eared herbivore escapes through fast hops.', 'Powerful rear legs propel rapid bounds.', 'Many species rely on burrows for shelter.', 'Constantly growing incisors grind plant material.', 'Eyes on the sides of the head detect threats.']}, 'deer': {'group': 'land_mammals', 'facts': ['A hoofed browser or grazer often has antlers.', 'Many males shed and regrow bony headgear annually.', 'Herds move between feeding and sheltering sites.', 'Alert ears and rapid sprints aid predator avoidance.', 'Young animals may hide in vegetation at first.']}, 'bear': {'group': 'land_mammals', 'facts': ['A heavy omnivore uses strong paws to forage.', 'Some populations spend winter in prolonged dormancy.', 'Thick fur and stored fat aid cold-weather survival.', 'Individuals may catch fish in shallow streams.', 'Cubs accompany their mothers while learning food sources.']}, 'horse': {'group': 'land_mammals', 'facts': ['A large hoofed grazer is domesticated for riding.', 'Long legs support sustained movement over grassland.', 'Herd members communicate through posture and calls.', 'A mane runs along the upper neck.', 'Hooves require regular care in human-managed settings.']}, 'eagle': {'group': 'birds', 'facts': ['A powerful raptor soars on rising air.', 'Hooked beaks tear food into pieces.', 'Large talons grip captured animals.', 'Keen vision helps locate prey from high above.', 'Many pairs maintain extensive nesting territories.']}, 'penguin': {'group': 'birds', 'facts': ['A flightless seabird uses flippers to swim.', 'Dense feathers insulate against chilly seawater.', 'Colonies gather on rocky or icy coasts.', 'Upright walking contrasts with agile underwater hunting.', 'Adults may take turns guarding an egg.']}, 'owl': {'group': 'birds', 'facts': ['A nocturnal hunter often flies almost silently.', 'Forward-facing eyes aid depth judgment.', 'Soft-edged flight feathers reduce noise.', 'A flexible neck allows a wide viewing angle.', 'Regurgitated pellets contain undigested bones.']}, 'parrot': {'group': 'birds', 'facts': ['A colorful bird has a curved strong beak.', 'Some species imitate human speech sounds.', 'Grasping feet manipulate seeds and fruit.', 'Social flocks communicate with loud calls.', 'Long-lived individuals can learn complicated routines.']}, 'swan': {'group': 'birds', 'facts': ['A large waterbird has an elegant curved neck.', 'Broad webbed feet propel it across lakes.', 'Pairs often defend nesting areas vigorously.', 'Long necks reach submerged aquatic plants.', 'Young birds travel close to protective adults.']}, 'flamingo': {'group': 'birds', 'facts': ['A wading bird uses a specialized filter-feeding bill.', 'Pink hues come partly from pigments in food.', 'Large groups gather in shallow salty water.', 'One-legged resting is frequently observed.', 'Head-down feeding strains tiny organisms from water.']}, 'peacock': {'group': 'birds', 'facts': ['A male gamebird spreads a dramatic tail display.', 'Iridescent eyespots decorate long ornamental plumes.', 'Loud calls carry through woodland edges.', 'Courtship involves fanning and vibrating feathers.', 'The female has comparatively subdued coloration.']}, 'sparrow': {'group': 'birds', 'facts': ['A small songbird thrives near human settlements.', 'Short conical bills crack seeds efficiently.', 'Flocks gather around bushes and feeders.', 'Quick hopping movements are common on the ground.', 'Simple cup-shaped nests fit into sheltered spaces.']}, 'woodpecker': {'group': 'birds', 'facts': ['A tree-climbing bird drums against wood.', 'A rigid tail braces the body on trunks.', 'Chisel-like bills excavate cavities.', 'Long tongues reach insects within bark.', 'Rhythmic tapping can also signal territorial claims.']}, 'albatross': {'group': 'birds', 'facts': ['A seabird glides over oceans on long wings.', 'Dynamic soaring reduces the need for flapping.', 'Individuals travel immense distances between landfalls.', 'Breeding pairs nest on remote oceanic islands.', 'A single chick may require prolonged parental care.']}, 'dolphin': {'group': 'marine_animals', 'facts': ['An intelligent marine mammal breathes at the surface.', 'Echolocation helps detect underwater objects.', 'Coordinated groups chase schools of fish.', 'A streamlined body supports rapid swimming.', 'Whistles and clicks carry social information.']}, 'shark': {'group': 'marine_animals', 'facts': ['A cartilaginous fish senses electrical fields.', 'Multiple rows of teeth can replace lost ones.', 'Some species must keep moving for water flow.', 'A dorsal fin often cuts above the surface.', 'Hunting relies on smell and specialized sensory organs.']}, 'whale': {'group': 'marine_animals', 'facts': ['A gigantic aquatic mammal surfaces to breathe.', 'Some species sieve krill through baleen plates.', 'Long migrations link feeding and breeding grounds.', 'Low-frequency calls can travel far underwater.', 'Thick blubber stores energy and provides insulation.']}, 'octopus': {'group': 'marine_animals', 'facts': ['A soft-bodied hunter has eight flexible arms.', 'Color-changing skin helps with camouflage.', 'Suckers manipulate shells and other objects.', 'The animal can squeeze through tiny openings.', 'Ink clouds may distract approaching predators.']}, 'jellyfish': {'group': 'marine_animals', 'facts': ['A gelatinous drifter propels itself by pulsing.', 'Trailing tentacles often bear stinging cells.', 'Ocean currents move these translucent animals.', 'A bell-shaped body lacks a conventional skeleton.', 'Some species glow in darkness.']}, 'seahorse': {'group': 'marine_animals', 'facts': ['A small fish swims upright in sea grass.', 'A curled tail grips stems underwater.', 'Males carry developing young in a brood pouch.', 'Tube-shaped mouths suck up tiny crustaceans.', 'Armor-like plates cover its slender body.']}, 'seal': {'group': 'marine_animals', 'facts': ['A marine mammal rests on rocks and ice.', 'Streamlined bodies dive for fish.', 'Whiskers detect water movements from nearby prey.', 'Thick fat helps conserve heat in cold seas.', 'Young pups nurse on beaches or frozen platforms.']}, 'crab': {'group': 'marine_animals', 'facts': ['A hard-shelled crustacean often walks sideways.', 'Two claws grip food and defend territory.', 'Molting allows the body to grow.', 'Gills enable breathing in aquatic environments.', 'Some kinds dig burrows in tidal sand.']}, 'starfish': {'group': 'marine_animals', 'facts': ['A seafloor animal has radiating arms.', 'Tube feet grip rocks and move the body.', 'Certain species regenerate damaged limbs.', 'Some feed by extending their stomach outside.', 'Rough spines cover the upper surface.']}, 'lobster': {'group': 'marine_animals', 'facts': ['A large crustacean shelters in rocky crevices.', 'Powerful claws can crush shells.', 'Antennae sense the nearby environment.', 'A hard exoskeleton must be shed as it grows.', 'The tail curls sharply to propel escape.']}, 'butterfly': {'group': 'insects_and_small_animals', 'facts': ['A winged insect begins life as a caterpillar.', 'Colored scales cover broad wings.', 'A coiled proboscis drinks flower nectar.', 'The pupal stage transforms the body dramatically.', 'Many species migrate or overwinter seasonally.']}, 'moth': {'group': 'insects_and_small_animals', 'facts': ['A winged insect is often active at night.', 'Feathery antennae detect faint chemical signals.', 'Many species are attracted to artificial lamps.', 'Larvae may feed on leaves or textiles.', 'Camouflaged wings blend into bark or foliage.']}, 'bee': {'group': 'insects_and_small_animals', 'facts': ['A flying pollinator gathers nectar and pollen.', 'Workers communicate food locations by movement.', 'Some colonies build wax combs for their young.', 'Hairy bodies carry pollen between blossoms.', 'A sting can defend the nest against threats.']}, 'ant': {'group': 'insects_and_small_animals', 'facts': ['A social insect lives in organized colonies.', 'Workers follow chemical trails to food.', 'Underground chambers shelter developing larvae.', 'Specialized individuals divide foraging and defense.', 'Many species carry loads heavier than themselves.']}, 'spider': {'group': 'insects_and_small_animals', 'facts': ['An eight-legged predator produces silk.', 'Webs can trap flying insects.', 'Venom subdues small prey.', 'Some species actively chase instead of waiting.', 'External digestion helps consume captured food.']}, 'scorpion': {'group': 'insects_and_small_animals', 'facts': ['A desert arthropod has a curved stinging tail.', 'Large pincers grasp prey.', 'Fluorescence appears under ultraviolet light.', 'Many individuals hide beneath rocks during daytime.', 'Sensory hairs detect vibrations in the ground.']}, 'beetle': {'group': 'insects_and_small_animals', 'facts': ['A six-legged insect has hardened wing covers.', 'Protective cases shelter delicate flying wings.', 'Many species recycle decaying wood or dung.', 'Larvae often look unlike the adult form.', 'Some show spectacular horns or metallic colors.']}, 'dragonfly': {'group': 'insects_and_small_animals', 'facts': ['A fast flying insect hunts above ponds.', 'Two pairs of long wings move independently.', 'Large compound eyes detect moving prey.', 'Aquatic juveniles develop beneath the water.', 'Adults can hover and change direction abruptly.']}, 'mosquito': {'group': 'insects_and_small_animals', 'facts': ['A slender insect develops in standing water.', 'Some females seek blood for egg production.', 'A high-pitched buzz accompanies flight.', 'Larvae wriggle near the water surface.', 'Carbon dioxide and body heat guide host seeking.']}, 'cricket': {'group': 'insects_and_small_animals', 'facts': ['A jumping insect produces rhythmic chirps.', 'Hind legs propel long leaps.', 'Sound often comes from rubbing wing structures.', 'Individuals hide among grasses and stones.', 'Calls help attract mates after dark.']}, 'banana': {'group': 'fruits', 'facts': ['An elongated yellow fruit grows in hanging clusters.', 'The peel opens without cutting.', 'Commercial plants often reproduce without seeds.', 'Soft flesh becomes sweeter as it ripens.', 'Warm climates support the large leafy crop.']}, 'apple': {'group': 'fruits', 'facts': ['A crisp orchard fruit has a central seed core.', 'Trees bloom before producing their harvest.', 'Many cultivars develop red or green skins.', 'Firm flesh can be baked into pies.', 'Harvested fruit keeps well under refrigeration.']}, 'orange': {'group': 'fruits', 'facts': ['A round citrus fruit has segmented juicy flesh.', 'Aromatic oils occur in the rind.', 'Vitamin-rich juice is widely consumed at breakfast.', 'White pith separates peel from edible sections.', 'Evergreen groves thrive in warm subtropical climates.']}, 'lemon': {'group': 'fruits', 'facts': ['A sharply sour citrus fruit is often bright yellow.', 'Zest adds fragrance to cooked dishes.', 'Acidic juice helps prevent sliced produce browning.', 'Many fruits have pointed ends.', 'A squeeze provides tartness for drinks and sauces.']}, 'lime': {'group': 'fruits', 'facts': ['A small green citrus fruit has acidic juice.', 'The rind provides intense aromatic zest.', 'Wedges accompany many tropical beverages.', 'The fruit is often harvested before turning yellow.', 'Tart juice pairs with savory marinades.']}, 'grape': {'group': 'fruits', 'facts': ['Small round berries grow in bunches on vines.', 'Some cultivars are used for winemaking.', 'Thin skins surround juicy seed-bearing flesh.', 'Climbing plants are trained on trellises.', 'Dried berries become raisins.']}, 'peach': {'group': 'fruits', 'facts': ['A soft stone fruit has fuzzy skin.', 'A large hard pit protects the seed.', 'Fragrant flesh ripens during summer.', 'Blossoms appear on orchard branches in spring.', 'Tender slices bruise when handled roughly.']}, 'pear': {'group': 'fruits', 'facts': ['A tree fruit often has a narrow neck and broad base.', 'The flesh may have tiny gritty cells.', 'Many varieties soften after picking.', 'Poaching develops a delicate fragrance.', 'Orchard trees bear white spring blossoms.']}, 'pineapple': {'group': 'fruits', 'facts': ['A tropical fruit has rough patterned outer skin.', 'Spiky leaves form a crown at the top.', 'Sweet acidic flesh grows around a fibrous core.', 'A single fruit forms from fused flowers.', 'Fresh juice can tenderize some proteins.']}, 'strawberry': {'group': 'fruits', 'facts': ['A small red berry carries visible seeds outside.', 'Low plants send runners across beds.', 'White flowers precede soft fragrant fruit.', 'The harvest spoils quickly without cooling.', 'Sweet slices often top cakes and desserts.']}, 'carrot': {'group': 'vegetables', 'facts': ['A tapered orange root vegetable grows underground.', 'Feathery green foliage emerges above the soil.', 'Crunchy roots are commonly eaten raw.', 'Beta carotene contributes to the bright pigment.', 'Thin slices soften in soups and stews.']}, 'potato': {'group': 'vegetables', 'facts': ['A starchy edible tuber grows beneath the soil.', 'Sprouting eyes can produce new plants.', 'Boiling and baking soften dense white flesh.', 'Harvested tubers store energy for the plant.', 'Greened skin should not be eaten.']}, 'tomato': {'group': 'vegetables', 'facts': ['A juicy red garden fruit is used as a vegetable.', 'Numerous seeds sit in gel-filled chambers.', 'Vines are often supported with stakes.', 'Heating produces sauces with savory depth.', 'Green specimens gradually change color when ripe.']}, 'cucumber': {'group': 'vegetables', 'facts': ['A long green vegetable has watery crisp flesh.', 'Climbing vines bear yellow flowers.', 'Tiny seeds run along the soft center.', 'Thin slices cool salads and sandwiches.', 'Pickling preserves the harvest in brine.']}, 'onion': {'group': 'vegetables', 'facts': ['A layered bulb has a sharp aroma when cut.', 'Sulfur compounds make the eyes water.', 'Papery skin protects fleshy concentric layers.', 'Slow cooking creates a sweet brown flavor.', 'Bulbs can be stored in a dry pantry.']}, 'garlic': {'group': 'vegetables', 'facts': ['A pungent bulb divides into separate cloves.', 'Crushed cloves release strong sulfur aromas.', 'Individual segments are planted to grow new bulbs.', 'Roasting produces a mellow spreadable texture.', 'The plant belongs to the allium family.']}, 'broccoli': {'group': 'vegetables', 'facts': ['A green vegetable has clusters of unopened buds.', 'Thick stalks support branched florets.', 'Steaming softens the crisp stems.', 'Yellow flowers emerge if heads are left unharvested.', 'Leaves also contain edible nutrients.']}, 'cabbage': {'group': 'vegetables', 'facts': ['A leafy vegetable forms a tight round head.', 'Broad leaves overlap in dense layers.', 'Shredding produces the base of a slaw.', 'Fermentation creates a tangy preserved food.', 'Cool weather favors head development.']}, 'eggplant': {'group': 'vegetables', 'facts': ['A glossy purple vegetable has spongy flesh.', 'Tiny seeds remain embedded in the interior.', 'The skin becomes tender after roasting.', 'Warm soils support the flowering plant.', 'Slices absorb cooking oil readily.']}, 'pumpkin': {'group': 'vegetables', 'facts': ['A large orange squash has a hard outer rind.', 'A hollow center contains flat seeds.', 'Thick flesh is cooked into soups and pies.', 'Vines spread across garden ground.', 'Harvested fruit often decorates autumn displays.']}, 'bicycle': {'group': 'vehicles', 'facts': ['A two-wheeled vehicle moves by pedaling.', 'A chain commonly transfers power to a rear wheel.', 'Hand brakes slow the rotating rims or discs.', 'Balance and steering enable turning.', 'Riders often wear helmets for protection.']}, 'motorcycle': {'group': 'vehicles', 'facts': ['A powered two-wheeled road vehicle carries a rider.', 'A twist throttle controls engine power.', 'Leaning helps negotiate curves at speed.', 'Protective jackets reduce injuries in a fall.', 'Exhaust sound accompanies many fuel-driven models.']}, 'car': {'group': 'vehicles', 'facts': ['A four-wheeled road vehicle transports passengers.', 'Doors provide access to an enclosed cabin.', 'Steering wheels direct the front axle.', 'Seat belts restrain occupants during collisions.', 'A trunk holds luggage for journeys.']}, 'bus': {'group': 'vehicles', 'facts': ['A large road vehicle carries many seated passengers.', 'Fixed routes stop at designated locations.', 'Wide doors speed boarding and exit.', 'Public transit reduces the number of separate cars.', 'Standing riders often hold overhead rails.']}, 'train': {'group': 'vehicles', 'facts': ['Rail vehicles travel along fixed steel tracks.', 'Multiple carriages couple behind motive power.', 'Stations organize boarding along platforms.', 'Long routes transport people and freight.', 'Signals coordinate safe passage on shared lines.']}, 'tram': {'group': 'vehicles', 'facts': ['A street-running rail vehicle serves urban neighborhoods.', 'Tracks often share corridors with road traffic.', 'Overhead wires supply power to many systems.', 'Frequent stops enable short journeys.', 'The cabin typically has wide entry doors.']}, 'airplane': {'group': 'vehicles', 'facts': ['A fixed-wing aircraft flies using aerodynamic lift.', 'Engines provide forward thrust.', 'Airport runways support takeoff and landing.', 'Pressurized cabins carry travelers at altitude.', 'Flaps change wing performance at low speeds.']}, 'helicopter': {'group': 'vehicles', 'facts': ['A rotorcraft can hover over a single location.', 'Spinning overhead blades generate lift.', 'Vertical takeoff requires little runway space.', 'A tail rotor often balances turning forces.', 'Rescue teams use it to reach remote areas.']}, 'boat': {'group': 'vehicles', 'facts': ['A small watercraft floats on rivers or lakes.', 'Hull shape displaces water to provide buoyancy.', 'Paddles or engines produce forward motion.', 'A rudder or oar helps steer.', 'Passengers may wear personal flotation devices.']}, 'submarine': {'group': 'vehicles', 'facts': ['An underwater vessel can travel below the surface.', 'Ballast tanks change buoyancy for diving.', 'Thick hulls withstand external water pressure.', 'Sonar helps navigation without direct visibility.', 'Crews may remain submerged for extended periods.']}, 'piano': {'group': 'musical_instruments', 'facts': ['A keyboard instrument makes notes by striking strings.', 'Pedals sustain or alter the sound.', 'Black and white keys span multiple octaves.', 'A wooden soundboard amplifies vibrations.', 'Two hands can play independent melodic lines.']}, 'violin': {'group': 'musical_instruments', 'facts': ['A small string instrument is played beneath the chin.', 'A bow draws sound from four strings.', 'Fine finger placement changes the pitch.', 'A hollow wooden body resonates acoustically.', 'Orchestras commonly group many performers together.']}, 'guitar': {'group': 'musical_instruments', 'facts': ['A fretted string instrument is held against the torso.', 'Plucking or strumming sets strings vibrating.', 'Chords use several notes at once.', 'An acoustic wooden body amplifies vibrations.', 'Players often press strings along a fingerboard.']}, 'flute': {'group': 'musical_instruments', 'facts': ['A wind instrument makes sound across an air opening.', 'Fingered keys change the resonant air column.', 'It is often held horizontally while playing.', 'Bright high tones can carry over an orchestra.', 'Breath control shapes the musical phrase.']}, 'clarinet': {'group': 'musical_instruments', 'facts': ['A woodwind instrument uses a single vibrating reed.', 'The mouthpiece supports a flexible cane strip.', 'A cylindrical bore gives its characteristic tone.', 'Keys cover holes along a long tube.', 'The instrument has a broad low-to-high range.']}, 'saxophone': {'group': 'musical_instruments', 'facts': ['A metal woodwind uses a single-reed mouthpiece.', 'A curved bell projects its expressive sound.', 'Key pads open and close tone holes.', 'Jazz solos often feature this instrument.', 'Several sizes cover different registers.']}, 'trumpet': {'group': 'musical_instruments', 'facts': ['A brass instrument uses buzzing lips to create tones.', 'Three valves redirect air through tubing.', 'A flared bell projects a brilliant sound.', 'Players alter pitch through embouchure.', 'Fanfare melodies frequently use this instrument.']}, 'trombone': {'group': 'musical_instruments', 'facts': ['A brass instrument changes pitch using a long slide.', 'A large bell projects deep resonant tones.', 'Moving parallel tubes alters air-column length.', 'It often supports low orchestral harmonies.', 'Players coordinate arm movement with lip tension.']}, 'drum': {'group': 'musical_instruments', 'facts': ['A percussion instrument sounds when a membrane is struck.', 'Sticks or hands create rhythmic patterns.', 'Tension affects the pitch and timbre.', 'Different sizes produce varied low and high sounds.', "Repeated beats establish a song's pulse."]}, 'harp': {'group': 'musical_instruments', 'facts': ['A large frame instrument has many plucked strings.', 'The strings descend across an open triangular frame.', 'Pedals on concert versions change pitches.', 'Sweeping arpeggios create rippling effects.', 'Players use fingertips of both hands.']}, 'hammer': {'group': 'tools', 'facts': ['A striking tool has a heavy head and handle.', 'Repeated blows drive nails into wood.', 'A claw may pull embedded fasteners.', 'The grip transfers force from an arm swing.', 'Metal heads can damage delicate surfaces.']}, 'screwdriver': {'group': 'tools', 'facts': ['A hand tool rotates threaded fasteners.', 'Shaped tips fit matching slots in screw heads.', 'Torque transfers through a narrow shaft.', 'Interchangeable bits handle different patterns.', 'The handle is turned rather than struck.']}, 'wrench': {'group': 'tools', 'facts': ['A gripping tool turns nuts and bolts.', 'A shaped jaw engages flat-sided hardware.', 'Adjustable designs fit several sizes.', 'Leverage helps loosen stubborn connections.', 'Mechanics frequently use it around engines.']}, 'pliers': {'group': 'tools', 'facts': ['A hinged hand tool grips objects between jaws.', 'Squeezing two handles increases clamping force.', 'Some designs cut thin wires.', 'Serrated metal faces prevent slipping.', 'The pivot allows strong precise holds.']}, 'saw': {'group': 'tools', 'facts': ['A cutting tool has a line of sharp teeth.', 'Back-and-forth strokes remove material.', 'Different blades cut wood or metal.', 'The kerf leaves a narrow channel.', 'Guides can help keep the cut straight.']}, 'chisel': {'group': 'tools', 'facts': ['A narrow bladed tool carves or shapes solid material.', 'The cutting edge is guided into the surface.', 'A mallet may tap the handle.', 'Woodworkers use it for joints and grooves.', 'Sharpening preserves a clean controlled cut.']}, 'scissors': {'group': 'tools', 'facts': ['A hand tool cuts with two crossing blades.', 'Finger loops control the hinged handles.', 'Shearing action divides paper or fabric.', 'The pivot holds sharpened edges together.', 'Right-handed and left-handed models differ.']}, 'knife': {'group': 'tools', 'facts': ['A handheld cutting tool has one sharp blade.', 'Edge geometry determines slicing performance.', 'A handle keeps fingers behind the cutting edge.', 'Kitchen versions prepare food ingredients.', 'A protective sheath can cover the point.']}, 'drill': {'group': 'tools', 'facts': ['A powered or manual tool creates round holes.', 'A rotating bit removes material.', 'A chuck secures interchangeable attachments.', 'Speed settings suit different surfaces.', 'Some designs also drive threaded fasteners.']}, 'shovel': {'group': 'tools', 'facts': ['A long-handled tool moves soil or sand.', 'A broad blade enters loose ground.', 'Foot pressure can push the edge deeper.', 'Gardeners use it to dig planting holes.', 'Levering the handle lifts a loaded scoop.']}, 'refrigerator': {'group': 'home_appliances', 'facts': ['An insulated cabinet keeps food cold.', 'A compressor moves heat out of the interior.', 'Shelves organize perishables in compartments.', 'Door seals reduce the entry of warm air.', 'Thermostats regulate storage temperature.']}, 'oven': {'group': 'home_appliances', 'facts': ['An enclosed appliance cooks food with sustained heat.', 'Metal racks support trays and dishes.', 'Baking transforms dough into crusty bread.', 'Insulated walls limit heat loss.', 'Temperature dials control roasting conditions.']}, 'microwave': {'group': 'home_appliances', 'facts': ['A countertop appliance heats food using electromagnetic waves.', 'Rotating trays help distribute energy.', 'Short cycles rewarm leftovers rapidly.', 'A shielded door prevents radiation leakage.', 'Moist food warms by dielectric heating.']}, 'toaster': {'group': 'home_appliances', 'facts': ['A small kitchen appliance browns sliced bread.', 'Electric elements heat both sides.', 'A spring often raises the food when finished.', 'Browning settings change heating duration.', 'Crumbs collect in a removable tray.']}, 'blender': {'group': 'home_appliances', 'facts': ['A kitchen appliance mixes food with rapidly spinning blades.', 'A lidded jar contains liquids during operation.', 'Fruit and ice can become a smooth drink.', 'Different speeds change texture.', 'The motor drives a cutting assembly at the bottom.']}, 'vacuum': {'group': 'home_appliances', 'facts': ['A cleaning appliance removes dust with suction.', 'An electric motor moves air through a chamber.', 'Filters trap fine particles before exhaust.', 'Hose attachments reach narrow spaces.', 'A collection bag or bin holds debris.']}, 'washer': {'group': 'home_appliances', 'facts': ['A laundry appliance cleans clothing with water.', 'A rotating drum agitates fabric.', 'Detergent loosens soil and oils.', 'Spin cycles remove much of the remaining moisture.', 'Program settings select wash temperature and duration.']}, 'dryer': {'group': 'home_appliances', 'facts': ['A laundry appliance removes water from washed clothes.', 'Heated airflow passes through tumbling fabric.', 'A lint filter catches loose fibers.', 'Moisture sensors may stop the cycle automatically.', 'Venting or condensation manages damp exhaust.']}, 'dishwasher': {'group': 'home_appliances', 'facts': ['A kitchen appliance cleans plates with sprayed water.', 'Interior racks hold utensils and dishes.', 'Detergent dissolves food residue during cycles.', 'Hot rinses help with drying and sanitation.', 'A pump circulates water through spray arms.']}, 'fan': {'group': 'home_appliances', 'facts': ['An electric appliance circulates air with rotating blades.', 'Adjustable speeds change airflow strength.', 'A grille protects users from moving parts.', 'Oscillation spreads the breeze across a room.', 'Ventilation can improve comfort without cooling air itself.']}}

## 1. Frozen stimuli and distance computation

- The first **three** attributes are authored *training* description families; the last **two** are *evaluation* families, never used for Probe fit or comparison selection.
- Compare semantic distance using only the concept name and **first training gloss**, thus avoiding direct use of evaluation descriptions for selecting Ys.
- Choose three Y comparators per X: closest, median, farthest **within the same semantic category**. Exclude uninformative choices if distance separations are tiny, and record that decision.
- All pairs are selected and recorded **before** the Qwen model runs. Since 2 evaluation families/class are very few, the primary AUC is exploratory and has coarse resolution.

In [ ]:
assert len(BANK)==100
GROUPS=defaultdict(list)
for name,obj in BANK.items():
    assert len(obj['facts'])==5
    GROUPS[obj['group']].append(name)
assert len(GROUPS)==10 and all(len(x)==10 for x in GROUPS.values())
texts=[f"{name}. {obj['facts'][0]}" for name,obj in BANK.items()]
order=list(BANK)
embedder=SentenceTransformer(EMBEDDER_ID,device='cpu')
embeddings=embedder.encode(texts,normalize_embeddings=True,show_progress_bar=True)
distance=1-embeddings@embeddings.T
index={c:i for i,c in enumerate(order)}
comparators=[]
for group,members in sorted(GROUPS.items()):
    for x in members:
        candidates=sorted([y for y in members if y!=x],key=lambda y:(distance[index[x],index[y]],y))
        picks=[('near',candidates[0]),('medium',candidates[4]),('far',candidates[-1])]
        vals=[float(distance[index[x],index[y]]) for _,y in picks]
        for band,y in picks:
            comparators.append({'target':x,'comparator':y,'group':group,'band':band,'cosine_distance':float(distance[index[x],index[y]]),'near_mid_far_span':vals[-1]-vals[0]})
comparison_df=pd.DataFrame(comparators)
comparison_df.to_csv(OUT/'candidate_comparisons_preselected.csv',index=False)
assert len(comparison_df)==300
print('Candidates:',len(comparison_df),'unique unordered pairs:',len({tuple(sorted((r.target,r.comparator))) for r in comparison_df.itertuples()}))
print('Distance band summaries:')
print(comparison_df.groupby('band').cosine_distance.describe())
print('Smallest near-to-far span:',comparison_df.near_mid_far_span.min())

In [ ]:
# A single prompt per concept × semantic description × surface wrapper; 100 × 5 × 2 = 1000.
TRAIN_WRAPPERS=["A field note records: {fact}","The observation states: {fact}"]
EVAL_WRAPPERS=["An independent field report describes: {fact}","A separate account reports: {fact}"]
records=[]
for concept,obj in BANK.items():
    for family,fact in enumerate(obj['facts']):
        split='train' if family<3 else 'eval'
        wrappers=TRAIN_WRAPPERS if split=='train' else EVAL_WRAPPERS
        for w,frame in enumerate(wrappers):
            context=frame.format(fact=fact)
            instruction=f"Read the observation carefully.\n\nObservation: {context}\n\nRespond with one brief sentence about the observation."
            records.append({'concept':concept,'group':obj['group'],'family':family if split=='train' else family-3,'split':split,'wrapper':w,'fact':fact,'user_prompt':instruction})
prompts_df=pd.DataFrame(records)
assert len(prompts_df)==1000
assert not prompts_df.user_prompt.duplicated().any()
# Exact target-name leakage check; allow generic words only if no exact concept name token.
leaks=[]
for r in prompts_df.itertuples():
    if re.search(r'(?i)\b'+re.escape(r.concept)+r'\b',r.fact):
        leaks.append((r.concept,r.fact))
assert not leaks,f'Concept-name leakage: {leaks[:10]}'
prompts_df.to_csv(OUT/'stimulus_design.csv',index=False)
PROMPT_HASH=hashlib.sha256(prompts_df.to_csv(index=False).encode()).hexdigest()
CONFIG={'model':MODEL_ID,'revision':MODEL_REVISION,'embedding_model':EMBEDDER_ID,'concepts':100,'prompt_sha256':PROMPT_HASH,'layers':LAYERS,'position':POSITION,'seed':SEED,'batch_size':BATCH_SIZE}
print('Prompt hash:',PROMPT_HASH)

## 2. GPU extraction with resumable immutable checkpoints

Each checkpoint is validated for prompt/configuration identity. Existing remote checkpoints are **restored rather than regenerated** to avoid bitwise differences caused by computation/library versions. ZIP byte metadata never decides whether a checkpoint is valid; the checkpoint's verified arrays are used instead. Use a new run ID if changing stimuli or parameters.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=torch.float32,attn_implementation='eager').to('cuda').eval()
assert model.config.num_hidden_layers==24
HIDDEN=model.config.hidden_size

@torch.inference_mode()
def extract_one(user_prompt):
    # assistant_start is the representation at the end of a normal chat template
    tokens=tokenizer.apply_chat_template([{'role':'user','content':user_prompt}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
    states=model(input_ids=tokens,output_hidden_states=True,use_cache=False).hidden_states
    return np.stack([states[layer][0,-1].float().cpu().numpy() for layer in LAYERS],axis=0)

N=len(prompts_df)
BATCHES=math.ceil(N/BATCH_SIZE)
GROUPS_N=math.ceil(BATCHES/UPLOAD_EVERY)

def group_ids(g):return list(range(g*UPLOAD_EVERY,min((g+1)*UPLOAD_EVERY,BATCHES)))
def remote_zip(g):return f'{OUTPUT_PREFIX}/checkpoints/group_{g:03d}.zip'
def local_batch(b):return CACHE/f'batch_{b:03d}.npz'

def check_batch_bytes(b,blob):
    with np.load(io.BytesIO(blob),allow_pickle=False) as z:
        assert str(z['prompt_hash'].item())==PROMPT_HASH
        assert int(z['start'].item())==b*BATCH_SIZE
        assert int(z['stop'].item())==min(N,(b+1)*BATCH_SIZE)
        a=z['features']; assert a.shape==(min(N,(b+1)*BATCH_SIZE)-b*BATCH_SIZE,len(LAYERS),HIDDEN)
        assert np.isfinite(a).all()

def restore_group(g):
    blob=gh_download(remote_zip(g))
    if blob is None:return False
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        manifest=json.loads(z.read('checkpoint_config.json'))
        assert manifest==CONFIG,'Remote checkpoint config mismatch; set a NEW RUN_ID.'
        expected={f'batch_{b:03d}.npz' for b in group_ids(g)}|{'checkpoint_config.json'}
        assert set(z.namelist())==expected
        for b in group_ids(g):
            name=f'batch_{b:03d}.npz';data=z.read(name)
            check_batch_bytes(b,data)
            local_batch(b).write_bytes(data)
    print('Remote checkpoint restored/verified:',g)
    return True

def publish_group(g):
    if not all(local_batch(b).exists() for b in group_ids(g)):return False
    if restore_group(g):return True
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('checkpoint_config.json',json.dumps(CONFIG,sort_keys=True))
        for b in group_ids(g):
            blob=local_batch(b).read_bytes()
            check_batch_bytes(b,blob)
            z.writestr(f'batch_{b:03d}.npz',blob)
    gh_put_immutable(remote_zip(g),buf.getvalue(),f'Checkpoint Qwen 100 concept group {g:03d}')
    return True

for g in range(GROUPS_N):restore_group(g)
for b in tqdm(range(BATCHES),desc='Qwen hidden-state extraction'):
    path=local_batch(b)
    if path.exists():
        check_batch_bytes(b,path.read_bytes())
        continue
    start=b*BATCH_SIZE;stop=min(N,start+BATCH_SIZE)
    matrix=np.stack([extract_one(p) for p in prompts_df.user_prompt.iloc[start:stop]])
    np.savez_compressed(path,prompt_hash=np.array(PROMPT_HASH),start=np.array(start),stop=np.array(stop),features=matrix)
    if (b+1)%UPLOAD_EVERY==0 or b+1==BATCHES:
        publish_group(b//UPLOAD_EVERY)
for g in range(GROUPS_N):assert publish_group(g)
FEATURES=np.concatenate([np.load(local_batch(b),allow_pickle=False)['features'] for b in range(BATCHES)],axis=0)
assert FEATURES.shape==(N,len(LAYERS),HIDDEN)
print('Feature extraction complete:',FEATURES.shape)

## 3. Independent evaluation texts and paired comparison screening

Evaluation examples use two different authored semantic facts per concept, absent from the training facts and embedding inputs. This is a **new-text split**, but human evaluation is still needed. The main metric is macro AUC over 2 held-out semantic families per class (wrappers averaged before AUC); avoid treating wrapper duplicates as independent samples.

Train on three families for the primary Probe, assess sensitivity with each family removed (3 folds). To avoid spending excessive compute on hundreds of permutation fits, perform **exact 20 balanced family-level label allocations** on the primary Probe, for preliminarily acceptable contrasts. Given only 20 assignments, the minimum attainable exact upper-tail p is 0.05; use null-tail proportions descriptively, not as confirmatory p-values.

In [ ]:
# Deterministic index matrices: each concept × split × family corresponds to 2 prompts.
IDX={}
for (concept,split,family),sub in prompts_df.reset_index().groupby(['concept','split','family'],sort=False):
    IDX[(concept,split,int(family))]=sub['index'].to_numpy()

def family_features(concept,split,layer_index):
    nf=3 if split=='train' else 2
    return np.stack([FEATURES[IDX[(concept,split,f)],layer_index].mean(axis=0) for f in range(nf)])

def probe_fit(a,b):
    X=np.concatenate([a,b]); y=np.array([1]*len(a)+[0]*len(b))
    scaler=StandardScaler().fit(X)
    fit=LogisticRegression(C=0.1,solver='liblinear',max_iter=1000,random_state=SEED).fit(scaler.transform(X),y)
    return scaler,fit

def auc_of(scaler,fit,ax,bx):
    X=np.concatenate([ax,bx]);y=np.array([1]*len(ax)+[0]*len(bx))
    return float(roc_auc_score(y,fit.decision_function(scaler.transform(X))))

ordered_pairs=sorted(set(tuple(sorted((r.target,r.comparator))) for r in comparison_df.itertuples()))
scan=[]
for a,b in tqdm(ordered_pairs,desc='Unordered contrast screens'):
    row={'pair':f'{a}__{b}','a':a,'b':b,'group':BANK[a]['group']}
    for li,layer in enumerate(LAYERS):
        ta=family_features(a,'train',li);tb=family_features(b,'train',li)
        ea=family_features(a,'eval',li);eb=family_features(b,'eval',li)
        scaler,fit=probe_fit(ta,tb)
        auc=auc_of(scaler,fit,ea,eb)
        leave=[]
        for f in range(3):
            retain=[i for i in range(3) if i!=f]
            ss,ff=probe_fit(ta[retain],tb[retain]);leave.append(auc_of(ss,ff,ea,eb))
        row[f'L{layer}_auc']=auc
        row[f'L{layer}_lofo_min']=min(leave)
        row[f'L{layer}_lofo_mean']=float(np.mean(leave))
    scan.append(row)
scan_df=pd.DataFrame(scan)
scan_df['screen_pass']=(scan_df['L24_auc']>=.85)&(scan_df['L24_lofo_min']>=.75)&(scan_df['L23_auc']>=.75)
scan_df.to_csv(OUT/'pair_screening.csv',index=False)
print('Distinct unordered pairs:',len(scan_df),'screen_pass:',int(scan_df.screen_pass.sum()))

In [ ]:
# Exact primary-Probe label shuffle; 3 semantic training families for each class.
# 20 label splits include correct and inverted labels. The goal is a coarse negative control.
shuffle_rows=[]
for r in tqdm(scan_df.loc[scan_df.screen_pass].itertuples(),total=int(scan_df.screen_pass.sum()),desc='Exact label controls'):
    a,b=r.a,r.b; li=LAYERS.index(24)
    ta=family_features(a,'train',li);tb=family_features(b,'train',li)
    ea=family_features(a,'eval',li);eb=family_features(b,'eval',li)
    X=np.concatenate([ta,tb]);y_true=np.array([1]*3+[0]*3)
    scaler=StandardScaler().fit(X);X_std=scaler.transform(X)
    X_eval=scaler.transform(np.concatenate([ea,eb]));y_eval=np.array([1]*len(ea)+[0]*len(eb))
    null=[]
    for chosen in itertools.combinations(range(6),3):
        y=np.zeros(6,dtype=int);y[list(chosen)]=1
        fit=LogisticRegression(C=0.1,solver='liblinear',max_iter=1000,random_state=SEED).fit(X_std,y)
        null.append(float(roc_auc_score(y_eval,fit.decision_function(X_eval))))
    observed=float(r.L24_auc)
    shuffle_rows.append({'pair':r.pair,'shuffle_total':len(null),'shuffle_tail_fraction':float(np.mean(np.asarray(null)>=observed-1e-12)), 'shuffle_mean_auc':float(np.mean(null)), 'observed_auc':observed})
shuffles_df=pd.DataFrame(shuffle_rows,columns=['pair','shuffle_total','shuffle_tail_fraction','shuffle_mean_auc','observed_auc'])
shuffles_df.to_csv(OUT/'exact_shuffle_screened_pairs.csv',index=False)
scan_df=scan_df.merge(shuffles_df,on='pair',how='left')
# Exploratory combined flag, deliberately not called statistically confirmed.
scan_df['exploratory_candidate']=scan_df.screen_pass & (scan_df.shuffle_tail_fraction<=0.10)
scan_df.to_csv(OUT/'pair_screening_with_controls.csv',index=False)
print('Exploratory pairs:',int(scan_df.exploratory_candidate.sum()))

## 4. Distance-aware target selection (and no circular success claim)

A target is a **preliminary distance-aware candidate** only if it has at least two successful comparison Ys **in different distance bands**, with a third one reported regardless of outcome. A distance-band effect must be read alongside continuous cosine distances; the three band labels are ordinal rankings within a category, not numeric matching across targets. This is selection on evaluation outcomes, so it **requires independent replication**.

In [ ]:
rowmap=scan_df.set_index('pair')
rows=[]
for x,g in comparison_df.groupby('target',sort=True):
    trials=[]
    for r in g.itertuples():
        key='__'.join(sorted((x,r.comparator)))
        s=rowmap.loc[key]
        trials.append({'band':r.band,'comparison':r.comparator,'distance':float(r.cosine_distance),'auc':float(s.L24_auc),'pass':bool(s.exploratory_candidate)})
    passed=[t for t in trials if t['pass']]
    rows.append({'target':x,'group':BANK[x]['group'],'stable_comparators':len(passed),'stable_bands':','.join(sorted(t['band'] for t in passed)), 'comparators':', '.join(t['comparison'] for t in passed),'near_auc':next(t['auc'] for t in trials if t['band']=='near'),'medium_auc':next(t['auc'] for t in trials if t['band']=='medium'),'far_auc':next(t['auc'] for t in trials if t['band']=='far'),'two_bands_stable':len(set(t['band'] for t in passed))>=2})
target_df=pd.DataFrame(rows).sort_values(['two_bands_stable','stable_comparators','target'],ascending=[False,False,True])
target_df.to_csv(OUT/'target_candidates.csv',index=False)
print('Distinct targets with 2+ passing comparators across distance bands:',int(target_df.two_bands_stable.sum()),'/ 100')
print(target_df.head(30).to_string(index=False))

# Matched summaries preserve the pairing of each target across three distance conditions.
joined=comparison_df.copy();joined['pair']=joined.apply(lambda r:'__'.join(sorted((r.target,r.comparator))),axis=1)
joined=joined.merge(scan_df[['pair','L24_auc','exploratory_candidate']],on='pair',how='left')
joined.to_csv(OUT/'distance_by_target_contrast.csv',index=False)
summary=joined.groupby('band').agg(n=('L24_auc','size'),mean_auc=('L24_auc','mean'),median_auc=('L24_auc','median'),passing=('exploratory_candidate','sum'),mean_distance=('cosine_distance','mean')).reset_index()
summary.to_csv(OUT/'distance_band_summary.csv',index=False)
print(summary)

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
bands=['near','medium','far']
values=[joined.loc[joined.band==b,'L24_auc'].dropna().to_numpy() for b in bands]
ax.boxplot(values,tick_labels=bands)
ax.set_ylabel('L24 assistant_start ROC-AUC (2 eval families/class)')
ax.set_xlabel('Within-category comparator distance rank')
ax.set_ylim(0,1.03)
ax.set_title('Exploratory X-vs-Y decodability by comparator distance')
fig.tight_layout();fig.savefig(OUT/'auc_by_distance_band.png',dpi=170);plt.show()

## 5. Export results and commit immutable GitHub artifacts

Results include source stimulus rows, frozen distance choices, all contrast-level AUCs and controls, detailed target ranking, plots, manifest, and full ZIP. Output paths are run-specific and immutable; for new data use a new run ID. The notebook does not claim 20 stable targets before seeing the data.

In [ ]:
manifest={**CONFIG,'status':'COMPLETED_EXPLORATORY_SCREEN','created_utc':datetime.now(timezone.utc).isoformat(),'gpu':torch.cuda.get_device_name(0),'prompts':N,'unique_unordered_contrasts':len(scan_df),'directed_distance_comparators':len(comparison_df),'exact_shuffle_pairs':len(shuffles_df),'target_candidates_two_distance_bands':int(target_df.two_bands_stable.sum()),'distance_def':'1-cosine(name + first training description; all-mpnet-base-v2)','notes':['The distance selection is calculated before any Qwen extraction, and excludes evaluation descriptions.','Only 2 independent evaluation facts per concept; the ROC-AUC is low resolution and exploratory.','Descriptions require independent human validation of concept identity and semantic non-overlap.','AUC is pairwise relative decodability, not absolute concept activation.','Candidate selection uses evaluation results; confirm shortlisted concepts on fresh text before use.','Semantic-distance bands reflect within-category rank rather than calibrated absolute distance.','Shuffle has only 20 exhaustive label splits; do not treat as a high-powered statistical test.']}
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
summary_text=f"""# Qwen Concept Probe 100 concepts — exploratory screen\n\n- Concepts: 100 in 10 categories\n- Source model: {MODEL_ID} at {MODEL_REVISION}\n- Independent authored train/evaluation facts: 3 + 2 per concept\n- Prompts: {N}\n- Pair selection: 3 within-category cosine-distance ranks × 100 target concepts\n- Unique unordered Probe contrasts: {len(scan_df)}\n- Screen-passing unordered contrasts: {int(scan_df.screen_pass.sum())}\n- With exact label-shuffle tail <= 0.10: {int(scan_df.exploratory_candidate.sum())}\n- Distinct X targets with two or more distance-band-stable Ys: {int(target_df.two_bands_stable.sum())} (goal: ~20)\n\n## Most promising targets\n\n{target_df.head(35).to_markdown(index=False)}\n\n## Distance-band AUC (exploratory)\n\n{summary.to_markdown(index=False)}\n\n**Interpretation:** This is pair-relative Probe identity classification only, not suppression or absolute activation. Two evaluation fact families per concept are insufficient for confirmatory measurement claims. All candidates require fresh, independently authored evaluation texts before thesis main experiments.\n"""
(OUT/'analysis_summary.md').write_text(summary_text)
zip_path=OUT/'qwen_100concept_exploratory_results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=zip_path:z.write(p,p.name)
for filename in ['manifest.json','analysis_summary.md','target_candidates.csv','distance_band_summary.csv','pair_screening_with_controls.csv']:
    data=(OUT/filename).read_bytes();gh_put_immutable(f'{OUTPUT_PREFIX}/{filename}',data,'Record Qwen 100-concept exploratory screen: '+filename)
gh_put_immutable(f'{OUTPUT_PREFIX}/{zip_path.name}',zip_path.read_bytes(),'Save Qwen 100-concept full exploratory results')
print('DONE. GitHub:',f'https://github.com/{REPO}/tree/main/{OUTPUT_PREFIX}')